# Hate Speech Classification with DSPy

This notebook uses [DSPy](https://dspy.ai) to build an optimized hate speech classifier.

DSPy is a declarative framework that enables:
- **Structured Signatures**: Define input/output types instead of crafting prompts
- **Automatic Optimization**: Learn effective prompts and few-shot examples
- **Modular Design**: Compose complex AI systems from simple modules

## Prerequisites & Setup

### 1. Get an API Key
Sign up at [openrouter.ai/settings/keys](https://openrouter.ai/settings/keys).

### 2. Set your API key
Add to your `.env` file:
```bash
OPENROUTER_API_KEY=sk-or-v1-xxxxxxxxxxxxxx
```

### 3. Python Environment (using UV)
```bash
# Create venv with uv
uv venv .venv-dspy
source .venv-dspy/bin/activate

# Install dependencies
uv add dspy-ai python-dotenv datasets scikit-learn pandas tqdm ipykernel

# Register kernel for Jupyter
python -m ipykernel install --user --name=dspy --display-name="DSPy"
```

### 4. Select the "DSPy" kernel
In VS Code/Jupyter, select the kernel dropdown and choose **DSPy**.

In [ ]:
import os
import json
import random
from typing import Literal
from datetime import datetime
import csv

import dspy
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from dotenv import load_dotenv
from sklearn.metrics import classification_report, confusion_matrix

load_dotenv()
print(f"✅ DSPy version: {dspy.__version__}")

## 1. Configuration

Configure the language model via OpenRouter (OpenAI-compatible endpoint).

In [ ]:
# --- Configuration ---

# Model selection (via OpenRouter)
# Uncomment your preferred model:

# CHEAP MODELS (best value)
# MODEL_NAME = "openai/gpt-4o-mini"                   # GPT-4o mini - $0.15/1M input
MODEL_NAME = "anthropic/claude-3.5-haiku"            # Claude 3.5 Haiku - $0.80/1M
# MODEL_NAME = "google/gemini-2.0-flash-001"         # Gemini Flash - $0.10/1M
# MODEL_NAME = "deepseek/deepseek-chat"              # DeepSeek V3 - $0.14/1M

# Dataset configuration
TRAIN_SIZE = 200      # Number of training examples for optimization
DEV_SIZE = 50         # Number of examples for validation during optimization
TEST_SIZE = 200       # Number of test examples for final evaluation

# Optimization settings
MAX_BOOTSTRAPPED_DEMOS = 2  # Few-shot examples to learn
MAX_LABELED_DEMOS = 2       # Additional labeled demos
NUM_CANDIDATE_PROGRAMS = 8  # Programs to explore in random search

# Verify API key
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise ValueError("❌ OPENROUTER_API_KEY not found in .env file!")

print(f"✅ Using model: {MODEL_NAME}")
print(f"✅ Train: {TRAIN_SIZE}, Dev: {DEV_SIZE}, Test: {TEST_SIZE} examples")

In [ ]:
# Configure DSPy with OpenRouter (OpenAI-compatible endpoint)
lm = dspy.LM(
    model=f"openrouter/{MODEL_NAME}",
    api_key=OPENROUTER_API_KEY,
    api_base="https://openrouter.ai/api/v1",
    temperature=0.0,
    max_tokens=50
)

dspy.configure(lm=lm)

# Quick test
test_response = lm("Say 'hello' and nothing else")
print(f"✅ API connection successful: {test_response[0] if test_response else 'No response'}")

## 2. Load HateXplain Dataset

Load the dataset from local JSONL files and convert to DSPy Examples.

In [ ]:
# Label mappings
ID2NAME = {0: "hatespeech", 1: "normal", 2: "offensive"}
NAME2ID = {v: k for k, v in ID2NAME.items()}
CLASSES = ["hatespeech", "normal"]  # Binary classification

def get_label(annotators):
    """Majority vote from annotators."""
    if not annotators:
        return 1
    labels = annotators.get("label", []) if isinstance(annotators, dict) else []
    if not labels:
        return 1
    counts = {0: 0, 1: 0, 2: 0}
    for lab in labels:
        lab = int(lab)
        if lab in counts:
            counts[lab] += 1
    return max(counts, key=lambda k: (counts[k], 2-k))


def normalize_example(example):
    """Extract text and label from HateXplain example."""
    text = " ".join(example["post_tokens"]) if isinstance(example.get("post_tokens"), list) else ""
    label_id = get_label(example.get("annotators", {}))
    label_name = ID2NAME[label_id]
    return {
        "text": text.strip(),
        "label_id": label_id,
        "label": label_name
    }


# Load from local files
print("Loading HateXplain dataset from local files...")

data_files = {
    "train": "hatexplain/hatexplain_train.jsonl",
    "validation": "hatexplain/hatexplain_validation.jsonl",
    "test": "hatexplain/hatexplain_test.jsonl"
}

# Check files exist
for split, path in data_files.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Could not find {split} file at {path}")

ds = load_dataset("json", data_files=data_files)
print(f"✅ Loaded: {len(ds['train'])} train, {len(ds['validation'])} val, {len(ds['test'])} test")

In [ ]:
def create_dspy_examples(dataset, max_size=None):
    """Convert HateXplain dataset to DSPy Examples."""
    examples = []
    for item in dataset:
        normalized = normalize_example(item)
        
        # Skip offensive (binary classification) and empty text
        if normalized["label"] == "offensive" or not normalized["text"]:
            continue
        
        example = dspy.Example(
            text=normalized["text"],
            label=normalized["label"]
        ).with_inputs("text")
        
        examples.append(example)
    
    # Shuffle and limit
    random.Random(42).shuffle(examples)
    if max_size:
        examples = examples[:max_size]
    
    return examples


# Create train, dev, and test sets
trainset = create_dspy_examples(ds["train"], TRAIN_SIZE)
devset = create_dspy_examples(ds["validation"], DEV_SIZE)
testset = create_dspy_examples(ds["test"], TEST_SIZE)

print(f"\n📊 Dataset splits:")
print(f"   Train: {len(trainset)} examples")
print(f"   Dev:   {len(devset)} examples")
print(f"   Test:  {len(testset)} examples")

# Show class distribution
for name, data in [("Train", trainset), ("Test", testset)]:
    hate_count = sum(1 for x in data if x.label == "hatespeech")
    normal_count = len(data) - hate_count
    print(f"   {name} distribution: {hate_count} hatespeech, {normal_count} normal")

In [ ]:
# Preview examples
print("\n📝 Sample examples:")
for i, ex in enumerate(trainset[:3]):
    print(f"\n[{i+1}] Label: {ex.label}")
    print(f"    Text: {ex.text[:100]}..." if len(ex.text) > 100 else f"    Text: {ex.text}")

## 3. Define DSPy Classification Module

Create a structured signature for hate speech classification.

In [ ]:
class HateSpeechClassify(dspy.Signature):
    """Classify social media text as hate speech or normal content.
    
    Hate speech includes content that promotes violence, discrimination, 
    or disparagement against protected groups (race, religion, gender, etc.).
    Normal content is everything else, even if crude or offensive in tone.
    """
    
    text: str = dspy.InputField(desc="Social media text to classify")
    label: Literal["hatespeech", "normal"] = dspy.OutputField(desc="Classification label")


# Create classifiers
# 1. Simple zero-shot predictor
predict_classifier = dspy.Predict(HateSpeechClassify)

# 2. Chain-of-thought classifier (with reasoning)
cot_classifier = dspy.ChainOfThought(HateSpeechClassify)

print("✅ DSPy classifiers defined:")
print("   - predict_classifier: Zero-shot prediction")
print("   - cot_classifier: Chain-of-thought (with reasoning)")

In [ ]:
# Test the classifiers
test_texts = [
    "I love everyone regardless of their background.",
    "Those people should all go back where they came from.",
    "Great weather today!"
]

print("🧪 Testing classifiers:\n")
for text in test_texts:
    result = predict_classifier(text=text)
    print(f"Text: {text}")
    print(f"  → {result.label}\n")

## 4. Define Evaluation Metric

Create a metric function for DSPy optimization and evaluation.

In [ ]:
def classification_metric(example, prediction, trace=None):
    """Metric for DSPy optimization: returns 1 if labels match, 0 otherwise."""
    # Normalize predictions
    pred_label = str(prediction.label).lower().strip()
    true_label = str(example.label).lower().strip()
    
    # Handle potential variations
    if "hate" in pred_label:
        pred_label = "hatespeech"
    elif "normal" in pred_label:
        pred_label = "normal"
    
    return 1.0 if pred_label == true_label else 0.0


def evaluate_classifier(classifier, dataset, desc="Evaluation"):
    """Evaluate a classifier and return detailed metrics."""
    y_true = []
    y_pred = []
    errors = 0
    
    for example in tqdm(dataset, desc=desc):
        try:
            result = classifier(text=example.text)
            pred = str(result.label).lower().strip()
            
            # Normalize prediction
            if "hate" in pred:
                pred = "hatespeech"
            elif "normal" in pred:
                pred = "normal"
            else:
                pred = "unknown"
                errors += 1
            
            y_true.append(example.label)
            y_pred.append(pred)
        except Exception as e:
            print(f"\n❌ Error: {e}")
            errors += 1
    
    return y_true, y_pred, errors


print("✅ Evaluation functions defined")

## 5. Baseline Evaluation

Evaluate the zero-shot classifiers before optimization.

In [ ]:
print("📊 Baseline Evaluation (Zero-Shot Predict)\n")

y_true_baseline, y_pred_baseline, errors_baseline = evaluate_classifier(
    predict_classifier, testset, desc="Baseline evaluation"
)

# Filter out unknowns for metrics
valid_mask = [p != "unknown" for p in y_pred_baseline]
y_true_valid = [y for y, v in zip(y_true_baseline, valid_mask) if v]
y_pred_valid = [y for y, v in zip(y_pred_baseline, valid_mask) if v]

if y_true_valid:
    report_baseline = classification_report(
        y_true_valid, y_pred_valid, 
        target_names=CLASSES, 
        output_dict=True, 
        zero_division=0
    )
    cm_baseline = confusion_matrix(y_true_valid, y_pred_valid, labels=CLASSES)
    
    print(f"\n📈 Baseline Results:")
    print(f"   Accuracy:  {report_baseline['accuracy']*100:.2f}%")
    print(f"   Macro F1:  {report_baseline['macro avg']['f1-score']*100:.2f}%")
    print(f"   Hate F1:   {report_baseline['hatespeech']['f1-score']*100:.2f}%")
    print(f"   Normal F1: {report_baseline['normal']['f1-score']*100:.2f}%")
    print(f"   Errors:    {errors_baseline}")

## 6. DSPy Optimization

Use BootstrapFewShot to learn effective few-shot demonstrations.

In [ ]:
print("🚀 Starting DSPy Optimization with BootstrapFewShot...\n")

# BootstrapFewShot optimizer
optimizer = dspy.BootstrapFewShot(
    metric=classification_metric,
    max_bootstrapped_demos=MAX_BOOTSTRAPPED_DEMOS,
    max_labeled_demos=MAX_LABELED_DEMOS
)

# Compile (optimize) the classifier
optimized_classifier = optimizer.compile(
    dspy.Predict(HateSpeechClassify),
    trainset=trainset
)

print("\n✅ Optimization complete!")
print(f"   Learned {len(optimized_classifier.demos)} demonstrations")

In [ ]:
# Show learned demonstrations
print("📝 Learned few-shot demonstrations:\n")
for i, demo in enumerate(optimized_classifier.demos[:3]):
    print(f"[Demo {i+1}]")
    text_preview = demo.text[:80] + "..." if len(demo.text) > 80 else demo.text
    print(f"  Text: {text_preview}")
    print(f"  Label: {demo.label}\n")

## 7. Evaluate Optimized Classifier

In [ ]:
print("📊 Optimized Classifier Evaluation\n")

y_true_opt, y_pred_opt, errors_opt = evaluate_classifier(
    optimized_classifier, testset, desc="Optimized evaluation"
)

# Filter out unknowns for metrics
valid_mask_opt = [p != "unknown" for p in y_pred_opt]
y_true_valid_opt = [y for y, v in zip(y_true_opt, valid_mask_opt) if v]
y_pred_valid_opt = [y for y, v in zip(y_pred_opt, valid_mask_opt) if v]

if y_true_valid_opt:
    report_opt = classification_report(
        y_true_valid_opt, y_pred_valid_opt, 
        target_names=CLASSES, 
        output_dict=True, 
        zero_division=0
    )
    cm_opt = confusion_matrix(y_true_valid_opt, y_pred_valid_opt, labels=CLASSES)

In [ ]:
# Pretty print results comparison
print("\n" + "═" * 60)
print("  📊 DSPY CLASSIFICATION RESULTS")
print("═" * 60)
print(f"\n  Model: {MODEL_NAME}")
print(f"  Samples: {len(y_true_valid_opt)} valid / {len(testset)} total")

print("\n┌" + "─" * 58 + "┐")
print("│" + " COMPARISON: BASELINE vs OPTIMIZED".center(58) + "│")
print("├" + "─" * 29 + "┬" + "─" * 28 + "┤")
print("│" + " Metric".center(15) + "│" + " Baseline".center(13) + "│" + " Optimized".center(13) + "│" + " Change".center(13) + "│")
print("├" + "─" * 15 + "┼" + "─" * 13 + "┼" + "─" * 13 + "┼" + "─" * 13 + "┤")

for metric_name, key in [("Accuracy", "accuracy"), ("Macro F1", "macro avg")]:
    if key == "accuracy":
        baseline_val = report_baseline[key] * 100
        opt_val = report_opt[key] * 100
    else:
        baseline_val = report_baseline[key]["f1-score"] * 100
        opt_val = report_opt[key]["f1-score"] * 100
    
    change = opt_val - baseline_val
    change_str = f"+{change:.1f}%" if change >= 0 else f"{change:.1f}%"
    print(f"│  {metric_name:<12} │  {baseline_val:>6.2f}%   │  {opt_val:>6.2f}%   │  {change_str:>8}   │")

print("└" + "─" * 15 + "┴" + "─" * 13 + "┴" + "─" * 13 + "┴" + "─" * 13 + "┘")

print("\n┌" + "─" * 58 + "┐")
print("│" + " PER-CLASS METRICS (OPTIMIZED)".center(58) + "│")
print("├" + "─" * 14 + "┬" + "─" * 14 + "┬" + "─" * 14 + "┬" + "─" * 13 + "┤")
print("│" + "  Class".center(14) + "│" + "  Precision".center(14) + "│" + "  Recall".center(14) + "│" + "  F1-Score".center(13) + "│")
print("├" + "─" * 14 + "┼" + "─" * 14 + "┼" + "─" * 14 + "┼" + "─" * 13 + "┤")
for class_name in CLASSES:
    m = report_opt[class_name]
    print(f"│  {class_name:<11} │  {m['precision']*100:>6.2f}%    │  {m['recall']*100:>6.2f}%    │  {m['f1-score']*100:>5.2f}%   │")
print("└" + "─" * 14 + "┴" + "─" * 14 + "┴" + "─" * 14 + "┴" + "─" * 13 + "┘")

print("\n┌" + "─" * 38 + "┐")
print("│" + " CONFUSION MATRIX".center(38) + "│")
print("├" + "─" * 38 + "┤")
print("│              Predicted".ljust(38) + "│")
print("│           hatespeech   normal".ljust(38) + "│")
print("├" + "─" * 38 + "┤")
print(f"│  Actual hatespeech │  {cm_opt[0][0]:>5}   │  {cm_opt[0][1]:>5}   │")
print(f"│         normal     │  {cm_opt[1][0]:>5}   │  {cm_opt[1][1]:>5}   │")
print("└" + "─" * 38 + "┘")

## 8. Save Results

In [ ]:
# Save results to model_scores.csv
results_dir = "results"
os.makedirs(results_dir, exist_ok=True)
results_file = os.path.join(results_dir, "model_scores.csv")

result_entry = {
    "timestamp": datetime.now().isoformat(),
    "model_name": f"dspy-{MODEL_NAME}",
    "model_type": "dspy (optimized)", 
    "task": "binary",
    "epochs": "N/A",
    "batch_size": "N/A",
    "learning_rate": "N/A",
    "max_length": "N/A",
    "accuracy": round(report_opt["accuracy"], 4),
    "macro_f1": round(report_opt["macro avg"]["f1-score"], 4),
    "macro_precision": round(report_opt["macro avg"]["precision"], 4),
    "macro_recall": round(report_opt["macro avg"]["recall"], 4),
    "hatespeech_f1": round(report_opt["hatespeech"]["f1-score"], 4),
    "hatespeech_precision": round(report_opt["hatespeech"]["precision"], 4),
    "hatespeech_recall": round(report_opt["hatespeech"]["recall"], 4),
    "normal_f1": round(report_opt["normal"]["f1-score"], 4),
    "normal_precision": round(report_opt["normal"]["precision"], 4),
    "normal_recall": round(report_opt["normal"]["recall"], 4),
    "cm_hate_as_hate": int(cm_opt[0][0]),
    "cm_hate_as_normal": int(cm_opt[0][1]),
    "cm_normal_as_hate": int(cm_opt[1][0]),
    "cm_normal_as_normal": int(cm_opt[1][1]),
    "test_samples": len(y_true_valid_opt),
}

file_exists = os.path.exists(results_file)
with open(results_file, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=result_entry.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(result_entry)

print("\n" + "═" * 60)
print(f"  ✅ Results saved to: {results_file}")
print("═" * 60)

In [ ]:
# Save optimized program for later use
program_path = f"dspy_programs/hate_speech_{MODEL_NAME.replace('/', '_')}.json"
os.makedirs("dspy_programs", exist_ok=True)

optimized_classifier.save(program_path)
print(f"✅ Optimized program saved to: {program_path}")

# To load later:
# loaded_classifier = dspy.Predict(HateSpeechClassify)
# loaded_classifier.load(program_path)

## 9. (Optional) Advanced Optimization with MIPROv2

MIPROv2 is a more advanced optimizer that also generates optimized instructions.
⚠️ **Warning**: This uses significantly more API calls.

In [ ]:
# Uncomment to run MIPROv2 optimization (uses many more API calls)
'''
print("🚀 Starting MIPROv2 Optimization (Advanced)...\n")

mipro_optimizer = dspy.MIPROv2(
    metric=classification_metric,
    auto="light",  # Options: "light", "medium", "heavy"
    num_threads=4
)

mipro_classifier = mipro_optimizer.compile(
    dspy.Predict(HateSpeechClassify),
    trainset=trainset,
    max_bootstrapped_demos=MAX_BOOTSTRAPPED_DEMOS,
    max_labeled_demos=MAX_LABELED_DEMOS
)

print("\n✅ MIPROv2 Optimization complete!")

# Evaluate MIPROv2 classifier
y_true_mipro, y_pred_mipro, errors_mipro = evaluate_classifier(
    mipro_classifier, testset, desc="MIPROv2 evaluation"
)
'''
print("💡 Uncomment the code above to run MIPROv2 optimization")

## 10. Interactive Testing

In [ ]:
def classify_text(text):
    """Classify a single text using the optimized classifier."""
    result = optimized_classifier(text=text)
    return result.label

# Interactive examples
print("🧪 Interactive Testing:\n")

test_examples = [
    "I can't believe how stupid these people are",
    "Just had the best pizza ever! 🍕",
    "All [group] should be deported immediately",
    "The weather is nice today",
    "This policy will hurt working families"
]

for text in test_examples:
    label = classify_text(text)
    emoji = "🚨" if label == "hatespeech" else "✅"
    print(f"{emoji} [{label}] {text}")